# Lab Work - 4.2

# Q1. K-Fold Cross Validation

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()
X = iris.data[:15]
y = iris.target[:15]

print('Dataset Size:', len(X))
print('Features Shape:', X.shape)

In [ ]:
kf = KFold(n_splits=5, shuffle=False)

accuracies = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    accuracies.append(acc)

    print(f'Fold {fold}')
    print('Train Indices:', train_idx)
    print('Test Indices :', test_idx)
    print('Accuracy     :', acc)
    print('-'*50)

In [ ]:
mean_cv = np.mean(accuracies)
std_cv = np.std(accuracies)

print('Mean CV Accuracy:', mean_cv)
print('Standard Deviation:', std_cv)

### Comparison with Train/Test Split

K-Fold generally provides a more reliable estimate because every sample is used for both training and testing.

# Q2. Stratified K-Fold

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Imbalanced Dataset
X_imb = np.arange(15).reshape(-1,1)
y_imb = np.array([0]*10 + [1]*5)

print('Class Distribution')
print(pd.Series(y_imb).value_counts())

In [ ]:
skf = StratifiedKFold(n_splits=3)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_imb, y_imb), start=1):
    print(f'Fold {fold}')

    train_classes = pd.Series(y_imb[train_idx]).value_counts()
    test_classes = pd.Series(y_imb[test_idx]).value_counts()

    print('Train Distribution')
    print(train_classes)

    print('Test Distribution')
    print(test_classes)

    print('-'*50)

### Observation

Stratified K-Fold preserves the original class ratio across all folds.

# Q4. LOOCV and Time Series Split

In [ ]:
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

x = np.array([1,2,3,4,5]).reshape(-1,1)
y = np.array([2,4,5,4,5])

loo = LeaveOneOut()

errors = []

for train_idx, test_idx in loo.split(x):
    X_train, X_test = x[train_idx], x[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = LinearRegression()
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    errors.append((y_test[0] - pred[0])**2)

loocv_mse = np.mean(errors)

print('LOOCV MSE =', loocv_mse)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

X_time = np.arange(1,13).reshape(-1,1)

tscv = TimeSeriesSplit(n_splits=3)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X_time), start=1):
    print(f'Fold {fold}')
    print('Train:', train_idx)
    print('Test :', test_idx)
    print('-'*50)

### Why Time Series Uses Expanding Windows

Future observations should never be used to predict the past.
Therefore training data expands forward through time.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,6))

for i, (train_idx, test_idx) in enumerate(tscv.split(X_time)):
    ax.scatter(train_idx, [i]*len(train_idx), marker='s', s=200, label='Train' if i==0 else '')
    ax.scatter(test_idx, [i]*len(test_idx), marker='s', s=200, label='Test' if i==0 else '')

ax.set_title('Time Series Split')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Fold')
ax.legend()
plt.show()

# Q4. Theory Questions

## Q4.1 What problem does Cross Validation solve?

A single train/test split may accidentally give an optimistic or pessimistic estimate depending on which samples are selected.

Cross Validation reduces this randomness by evaluating the model on multiple train/test partitions.

## Q4.2 Why does LOOCV have high variance but low bias?

- Training set is almost the entire dataset.
- Therefore bias is very low.
- Test set contains only one sample.
- Performance varies strongly across samples.
- Hence variance becomes high.

## Q4.3 When is Stratified K-Fold preferred?

1. Imbalanced datasets.
2. Medical diagnosis problems.
4. Fraud detection.
4. Spam detection.

It ensures every fold has similar class proportions.

## Q4.4 Why should Time Series never shuffle?

Time order contains important information.

Example:

If stock prices from 2025 are used to predict prices in 2024, the model gains information from the future.

This causes data leakage and unrealistic performance estimates.

# Conclusion

- K-Fold provides robust model evaluation.
- Stratified K-Fold preserves class balance.
- LOOCV uses maximum training data.
- Time Series Split respects chronological order.
- Choice depends on the nature of the dataset.